#### Mixed Variables Detection & Cleaning

In [1]:
import pandas as pd
import numpy as np

In [2]:
# ─── Inline Dataset: Ola Ride Records ───
data = {
    'ride_id':   [101, 102, 103, 104, 105, 106, 107, 108],
    'distance':  ['3.2 km', '5 km', 'N/A', '8.1 km',        # Number + Unit + Special
                  'Cancelled', '12.5 km', '2 km', '6.8 km'],
    'fare':      ['120', '200', 'Pending', '310',             # Number + String
                  'Refunded', '450', '90', '250'],
    'product_code': ['A100', 'B200', 'A150', 'C300',         # Code + Category mix
                     'B180', 'A200', 'C100', 'B250'],
}
df = pd.DataFrame(data)

print("Original Data:")
print(df)
print("\nData Types:")
print(df.dtypes)

Original Data:
   ride_id   distance      fare product_code
0      101     3.2 km       120         A100
1      102       5 km       200         B200
2      103        N/A   Pending         A150
3      104     8.1 km       310         C300
4      105  Cancelled  Refunded         B180
5      106    12.5 km       450         A200
6      107       2 km        90         C100
7      108     6.8 km       250         B250

Data Types:
ride_id          int64
distance        object
fare            object
product_code    object
dtype: object


In [3]:
df

,ride_id,distance,fare,product_code
0,101,3.2 km,120,A100
1,102,5 km,200,B200
2,103,N/A,Pending,A150
3,104,8.1 km,310,C300
4,105,Cancelled,Refunded,B180
5,106,12.5 km,450,A200
6,107,2 km,90,C100
7,108,6.8 km,250,B250


In [4]:
# ════════════════════════════════════════
# FIX 1: distance column — unit hatao, number nikalo
# Approach: str.replace() se ' km' hatao, phir pd.to_numeric se convert karo
# ════════════════════════════════════════
# Step 1: ' km' text hatao
df['distance_clean'] = df['distance'].str.replace(' km', '', case=False)
# Step 2: Special words (N/A, Cancelled) ko NaN kar do
special_vals = ['N/A', 'Cancelled', 'Unknown', '']
df['distance_clean'] = df['distance_clean'].apply(
    lambda x: np.nan if x in special_vals else x
)
# Step 3: pd.to_numeric — jo bhi valid number hai woh float banega, baaki NaN
df['distance_km'] = pd.to_numeric(df['distance_clean'], errors='coerce')

df[['distance', 'distance_km']]

,distance,distance_km
0,3.2 km,3.2
1,5 km,5.0
2,N/A,NaN
3,8.1 km,8.1
4,Cancelled,NaN
5,12.5 km,12.5
6,2 km,2.0
7,6.8 km,6.8


In [5]:
df

,ride_id,distance,fare,product_code,distance_clean,distance_km
0,101,3.2 km,120,A100,3.2,3.2
1,102,5 km,200,B200,5,5.0
2,103,N/A,Pending,A150,NaN,NaN
3,104,8.1 km,310,C300,8.1,8.1
4,105,Cancelled,Refunded,B180,NaN,NaN
5,106,12.5 km,450,A200,12.5,12.5
6,107,2 km,90,C100,2,2.0
7,108,6.8 km,250,B250,6.8,6.8


In [6]:
# ════════════════════════════════════════
# FIX 2: fare column — numeric convert karo
# pd.to_numeric se: '120' → 120.0 | 'Pending' → NaN (errors='coerce')
# ════════════════════════════════════════
df['fare_num'] = pd.to_numeric(df['fare'], errors='coerce')
# 'Pending', 'Refunded' automatically NaN ho jaate hain!

df[['fare', 'fare_num']]

,fare,fare_num
0,120,120.0
1,200,200.0
2,Pending,NaN
3,310,310.0
4,Refunded,NaN
5,450,450.0
6,90,90.0
7,250,250.0


In [7]:
# ════════════════════════════════════════
# FIX 3: product_code — category aur number alag karo
# 'A100' → category='A', number=100
# str[0] = pehla character (letter), str[1:] = baaki (number)
# ════════════════════════════════════════
df['product_category'] = df['product_code'].str[0]       # 'A100' → 'A'
df['product_num']      = pd.to_numeric(df['product_code'].str[1:])  # 'A100' → 100

df[['product_code', 'product_category', 'product_num']]

,product_code,product_category,product_num
0,A100,A,100
1,B200,B,200
2,A150,A,150
3,C300,C,300
4,B180,B,180
5,A200,A,200
6,C100,C,100
7,B250,B,250


In [8]:
# ════════════════════════════════════════
# FINAL RESULT
# ════════════════════════════════════════
# Temporary column drop karo
df.drop(columns=['distance_clean'], inplace=True)

print("\nNew Data Types:")
print(df[['distance_km','fare_num','product_num']].dtypes)

print("\nCleaned Dataset:")
df[['ride_id','distance_km','fare_num','product_category','product_num']]


New Data Types:
distance_km    float64
fare_num       float64
product_num      int64
dtype: object

Cleaned Dataset:


,ride_id,distance_km,fare_num,product_category,product_num
0,101,3.2,120.0,A,100
1,102,5.0,200.0,B,200
2,103,NaN,NaN,A,150
3,104,8.1,310.0,C,300
4,105,NaN,NaN,B,180
5,106,12.5,450.0,A,200
6,107,2.0,90.0,C,100
7,108,6.8,250.0,B,250


#### Date & Time Features Extraction

In [9]:
# ─── Inline Dataset: Swiggy Orders ───
data = {
    'order_id':  [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    'order_time': ['2024-01-15 13:30:00', '2024-03-22 20:15:00',
                   '2024-06-10 08:45:00', '2024-10-25 19:00:00',
                   '2024-12-25 12:00:00', '2024-04-07 22:30:00',
                   '2024-08-15 07:15:00', '2024-11-01 15:45:00'],
    'city':      ['Mumbai', 'Delhi', 'Bangalore', 'Hyderabad',
                  'Chennai', 'Pune', 'Kolkata', 'Ahmedabad'],
    'amount':    [350, 520, 180, 430, 850, 290, 620, 410],
}

df = pd.DataFrame(data)

df

,order_id,order_time,city,amount
0,1001,2024-01-15 13:30:00,Mumbai,350
1,1002,2024-03-22 20:15:00,Delhi,520
2,1003,2024-06-10 08:45:00,Bangalore,180
3,1004,2024-10-25 19:00:00,Hyderabad,430
4,1005,2024-12-25 12:00:00,Chennai,850
5,1006,2024-04-07 22:30:00,Pune,290
6,1007,2024-08-15 07:15:00,Kolkata,620
7,1008,2024-11-01 15:45:00,Ahmedabad,410


In [10]:
# ════════════════════════════════════════
# STEP 1: String → datetime convert karo
# ════════════════════════════════════════
df['order_time'] = pd.to_datetime(df['order_time'])
print("Data type after conversion:", df['order_time'].dtype)
# datetime64[ns]

Data type after conversion: datetime64[ns]


In [11]:
# ════════════════════════════════════════
# STEP 2: Basic date features nikalo
# ════════════════════════════════════════
df['year']     = df['order_time'].dt.year        # 2024
df['month']    = df['order_time'].dt.month       # 1-12
df['day']      = df['order_time'].dt.day         # 1-31
df['quarter']  = df['order_time'].dt.quarter     # 1-4
df['weekday']  = df['order_time'].dt.dayofweek   # 0=Mon, 6=Sun
df['week_num'] = df['order_time'].dt.isocalendar().week.astype(int)

# ════════════════════════════════════════
# STEP 3: Time features nikalo
# ════════════════════════════════════════
df['hour']   = df['order_time'].dt.hour    # 0-23
df['minute'] = df['order_time'].dt.minute  # 0-59


df[['order_time', 'year', 'month', 'day', 'hour', 'minute', 'quarter', 'weekday', 'week_num']]

,order_time,year,month,day,hour,minute,quarter,weekday,week_num
0,2024-01-15 13:30:00,2024,1,15,13,30,1,0,3
1,2024-03-22 20:15:00,2024,3,22,20,15,1,4,12
2,2024-06-10 08:45:00,2024,6,10,8,45,2,0,24
3,2024-10-25 19:00:00,2024,10,25,19,0,4,4,43
4,2024-12-25 12:00:00,2024,12,25,12,0,4,2,52
5,2024-04-07 22:30:00,2024,4,7,22,30,2,6,14
6,2024-08-15 07:15:00,2024,8,15,7,15,3,3,33
7,2024-11-01 15:45:00,2024,11,1,15,45,4,4,44


In [12]:
# ════════════════════════════════════════
# STEP 4: Derived / Business features banao
# ════════════════════════════════════════
# Weekend flag
df['is_weekend'] = df['weekday'].isin([5, 6]).astype(int)  # 5=Sat, 6=Sun

# Time of day category
def time_of_day(hour):
    if 5 <= hour < 12:   return 'Morning'
    elif 12 <= hour < 17: return 'Afternoon'
    elif 17 <= hour < 21: return 'Evening'
    else:                 return 'Night'

df['time_of_day'] = df['hour'].apply(time_of_day)

# Indian Season (roughly)
def indian_season(month):
    if month in [3, 4, 5]:   return 'Summer'
    elif month in [6, 7, 8, 9]: return 'Monsoon'
    elif month in [10, 11]:  return 'Autumn'
    else:                    return 'Winter'  # 12, 1, 2

df['season'] = df['month'].apply(indian_season)

# Days since a reference date
reference_date = pd.Timestamp('2024-01-01')
df['days_since_newyear'] = (df['order_time'] - reference_date).dt.days

# ════════════════════════════════════════
# RESULT
# ════════════════════════════════════════
print("\nExtracted Features:")

df[['order_id','month','weekday','hour','is_weekend', 'time_of_day','season','days_since_newyear']]


Extracted Features:


,order_id,month,weekday,hour,is_weekend,time_of_day,season,days_since_newyear
0,1001,1,0,13,0,Afternoon,Winter,14
1,1002,3,4,20,0,Evening,Summer,81
2,1003,6,0,8,0,Morning,Monsoon,161
3,1004,10,4,19,0,Evening,Autumn,298
4,1005,12,2,12,0,Afternoon,Winter,359
5,1006,4,6,22,1,Night,Summer,97
6,1007,8,3,7,0,Morning,Monsoon,227
7,1008,11,4,15,0,Afternoon,Autumn,305


#### Complete Case Analysis (CCA)

In [13]:
# ─── Inline Dataset: Student Exam Records ───
data = {
    'student':  ['Aarav','Priya','Ravi','Sneha','Karan',
                 'Divya','Mohit','Ananya','Vikram','Pooja'],
    'age':      [22, np.nan, 25, 23, np.nan, 21, 24, np.nan, 26, 20],
    'marks':    [85, 90, np.nan, 78, 88, np.nan, 72, 95, 80, np.nan],
    'city':     ['Mumbai','Delhi', np.nan,'Surat','Mumbai',
                 np.nan,'Delhi','Pune','Bangalore','Chennai'],
    'attendance':[90, 85, 80, np.nan, 92, 88, np.nan, 95, 78, 85],
}

df = pd.DataFrame(data)

print("Original Dataset:")
print(f"\nShape: {df.shape}")
print(f"\nMissing values per column:\n{df.isnull().sum()}")
print(f"\nRows with at least one NaN: {df.isnull().any(axis=1).sum()}")
df

Original Dataset:

Shape: (10, 5)

Missing values per column:
student       0
age           3
marks         3
city          2
attendance    2
dtype: int64

Rows with at least one NaN: 8


,student,age,marks,city,attendance
0,Aarav,22.0,85.0,Mumbai,90.0
1,Priya,NaN,90.0,Delhi,85.0
2,Ravi,25.0,NaN,NaN,80.0
3,Sneha,23.0,78.0,Surat,NaN
4,Karan,NaN,88.0,Mumbai,92.0
5,Divya,21.0,NaN,NaN,88.0
6,Mohit,24.0,72.0,Delhi,NaN
7,Ananya,NaN,95.0,Pune,95.0
8,Vikram,26.0,80.0,Bangalore,78.0
9,Pooja,20.0,NaN,Chennai,85.0


In [14]:
# ════════════════════════════════════════
# METHOD 1: All NaN rows hatao (basic CCA)
# ════════════════════════════════════════
df_cca = df.dropna()

print(f"\n--- Basic CCA (dropna all) ---")
print(f"Rows before: {len(df)} | Rows after: {len(df_cca)}")

df_cca


--- Basic CCA (dropna all) ---
Rows before: 10 | Rows after: 2


,student,age,marks,city,attendance
0,Aarav,22.0,85.0,Mumbai,90.0
8,Vikram,26.0,80.0,Bangalore,78.0


In [15]:
# ════════════════════════════════════════
# METHOD 2: Specific columns pe CCA
# sirf marks aur age missing ho toh hi drop karo
# ════════════════════════════════════════
df_cca2 = df.dropna(subset=['marks', 'age'])

print(f"\n--- CCA on specific columns (marks + age) ---")
print(f"Rows before: {len(df)} | Rows after: {len(df_cca2)}")

df_cca2


--- CCA on specific columns (marks + age) ---
Rows before: 10 | Rows after: 4


,student,age,marks,city,attendance
0,Aarav,22.0,85.0,Mumbai,90.0
3,Sneha,23.0,78.0,Surat,NaN
6,Mohit,24.0,72.0,Delhi,NaN
8,Vikram,26.0,80.0,Bangalore,78.0


In [16]:
# ════════════════════════════════════════
# METHOD 3: Threshold — agar 2+ columns NaN hain toh drop
# ════════════════════════════════════════
# thresh=4 means: kam se kam 4 non-NaN values chahiye row mein
df_cca4 = df.dropna(thresh=4)
print(f"\n--- CCA with thresh=4 ---")
print(f"Rows before: {len(df)} | Rows after: {len(df_cca4)}")

df_cca4


--- CCA with thresh=4 ---
Rows before: 10 | Rows after: 8


,student,age,marks,city,attendance
0,Aarav,22.0,85.0,Mumbai,90.0
1,Priya,NaN,90.0,Delhi,85.0
3,Sneha,23.0,78.0,Surat,NaN
4,Karan,NaN,88.0,Mumbai,92.0
6,Mohit,24.0,72.0,Delhi,NaN
7,Ananya,NaN,95.0,Pune,95.0
8,Vikram,26.0,80.0,Bangalore,78.0
9,Pooja,20.0,NaN,Chennai,85.0


In [17]:
# ════════════════════════════════════════
# BEFORE vs AFTER — Statistics Check
# ════════════════════════════════════════
print("\n--- Before vs After: marks column ---")
print(f"Mean BEFORE: {df['marks'].mean():.2f}")
print(f"Mean AFTER:  {df_cca['marks'].mean():.2f}")
print(f"Missing %:   {df.isnull().sum().sum() / df.size * 100:.1f}% original")
print(f"Missing %:   {df_cca.isnull().sum().sum() / df_cca.size * 100:.1f}% after CCA")


--- Before vs After: marks column ---
Mean BEFORE: 84.00
Mean AFTER:  82.50
Missing %:   20.0% original
Missing %:   0.0% after CCA


#### Complete Data Cleaning Pipeline — Mixed + DateTime + CCA

In [18]:
# ─── Inline Dataset: Ola Rides with all problem types ───
data = {
    'ride_id':   [101, 102, 103, 104, 105, 106, 107, 108, 109, 110],
    'booking_time': ['2024-03-15 08:30:00','2024-06-22 14:15:00',
                     '2024-10-05 20:00:00','2024-01-18 07:45:00',
                     None,                 '2024-08-11 18:30:00',
                     '2024-12-25 11:00:00','2024-04-30 16:20:00',
                     '2024-09-01 09:10:00','2024-07-14 21:45:00'],
    'distance':  ['5.2 km','8 km','N/A','12.5 km','3.8 km',
                  'Cancelled','6.1 km','9.3 km',None,'4.7 km'],
    'fare':      ['180','250','Pending','420','140',
                  'Refunded','210','310',None,'165'],
    'rating':    [4.5, None, 4.2, 4.8, 4.0, None, 4.7, 4.3, 4.6, None],
    'city':      ['Mumbai','Delhi',None,'Pune','Bangalore',
                  'Mumbai','Chennai',None,'Hyderabad','Kolkata'],
}
df = pd.DataFrame(data)

print("=== STEP 1: Original Dataset ===")
print(f"\nShape: {df.shape}")
print(f"Missing values:\n{df.isnull().sum()}")

df

=== STEP 1: Original Dataset ===

Shape: (10, 6)
Missing values:
ride_id         0
booking_time    1
distance        1
fare            1
rating          3
city            2
dtype: int64


,ride_id,booking_time,distance,fare,rating,city
0,101,2024-03-15 08:30:00,5.2 km,180,4.5,Mumbai
1,102,2024-06-22 14:15:00,8 km,250,NaN,Delhi
2,103,2024-10-05 20:00:00,N/A,Pending,4.2,None
3,104,2024-01-18 07:45:00,12.5 km,420,4.8,Pune
4,105,None,3.8 km,140,4.0,Bangalore
5,106,2024-08-11 18:30:00,Cancelled,Refunded,NaN,Mumbai
6,107,2024-12-25 11:00:00,6.1 km,210,4.7,Chennai
7,108,2024-04-30 16:20:00,9.3 km,310,4.3,None
8,109,2024-09-01 09:10:00,None,None,4.6,Hyderabad
9,110,2024-07-14 21:45:00,4.7 km,165,NaN,Kolkata


In [19]:
# ════════════════════════════════
# STEP 2: Date-Time Handle karo
# ════════════════════════════════
df['booking_time'] = pd.to_datetime(df['booking_time'], errors='coerce')
df['hour']         = df['booking_time'].dt.hour
df['month']        = df['booking_time'].dt.month
df['is_weekend']   = df['booking_time'].dt.dayofweek.isin([5,6]).astype('Int64')

print("\n=== STEP 2: DateTime Features Added ===")

df[['ride_id','hour','month','is_weekend']]


=== STEP 2: DateTime Features Added ===


,ride_id,hour,month,is_weekend
0,101,8.0,3.0,0
1,102,14.0,6.0,1
2,103,20.0,10.0,1
3,104,7.0,1.0,0
4,105,NaN,NaN,0
5,106,18.0,8.0,1
6,107,11.0,12.0,0
7,108,16.0,4.0,0
8,109,9.0,9.0,1
9,110,21.0,7.0,1


In [20]:
# ════════════════════════════════
# STEP 3: Mixed Variables Clean karo
# Simple approach: str.replace + pd.to_numeric
# ════════════════════════════════

# distance: ' km' text hatao, special values NaN, phir numeric convert
special_vals = ['N/A', 'Cancelled', 'Refunded', 'Pending', 'Unknown', '']

df['distance_clean'] = df['distance'].str.replace(' km', '', case=False)
df['distance_clean'] = df['distance_clean'].apply(
    lambda x: np.nan if x in special_vals else x
)
df['distance_km'] = pd.to_numeric(df['distance_clean'], errors='coerce')

# fare: sirf pd.to_numeric — 'Pending', 'Refunded' automatically NaN ho jaate
df['fare_num'] = pd.to_numeric(df['fare'], errors='coerce')

# Cleanup temporary column
df.drop(columns=['distance_clean'], inplace=True)

print("\n=== STEP 3: Mixed Variables Cleaned ===")

df[['ride_id','distance_km','fare_num']]


=== STEP 3: Mixed Variables Cleaned ===


,ride_id,distance_km,fare_num
0,101,5.2,180.0
1,102,8.0,250.0
2,103,NaN,NaN
3,104,12.5,420.0
4,105,3.8,140.0
5,106,NaN,NaN
6,107,6.1,210.0
7,108,9.3,310.0
8,109,NaN,NaN
9,110,4.7,165.0


In [21]:
# ════════════════════════════════
# STEP 4: CCA — Complete Case Analysis
# sirf core columns pe apply karo
# ════════════════════════════════
core_cols = ['distance_km', 'fare_num', 'rating', 'city', 'hour']
df_clean = df.dropna(subset=core_cols)

print(f"\n=== STEP 4: CCA Applied ===")
print(f"Rows before CCA: {len(df)}")
print(f"Rows after  CCA: {len(df_clean)}")
print(f"\nFinal Clean Dataset:")

df_clean[['ride_id','distance_km','fare_num',
                'rating','city','hour','month','is_weekend']]


=== STEP 4: CCA Applied ===
Rows before CCA: 10
Rows after  CCA: 3

Final Clean Dataset:


,ride_id,distance_km,fare_num,rating,city,hour,month,is_weekend
0,101,5.2,180.0,4.5,Mumbai,8.0,3.0,0
3,104,12.5,420.0,4.8,Pune,7.0,1.0,0
6,107,6.1,210.0,4.7,Chennai,11.0,12.0,0
